# **AutoML com Optuna**

## O que é AutoML?

- O **Automated Machine Learning** é também conhecido como **AutoML**;
- São ferramentas e técnicas que automatizam etapas do processo de desenvolvimento de modelos de ML/DL;
- Geralmente cobrem: normalização de dados, engenharia de features, treinamento com diferentes arquiteturas e hiperparâmetros, avaliação e comparação de resultados;

**Vantagens:**
- Reduz o tempo para obter resultados iniciais;
- Pode ser utilizado como baseline em diferentes tipos de problemas;
- Automatiza a busca por boas configurações de modelo;

**Desvantagens:**
- Pode demandar mais tempo computacional;
- Nem sempre encontra a melhor solução para problemas muito específicos;

---

## Por que usamos o Optuna neste notebook?

O notebook original utilizava a biblioteca **AutoKeras**. Porém, ao longo das últimas versões, o AutoKeras acumulou **incompatibilidades sérias com versões recentes do TensorFlow/Keras**, resultando em erros como:

```
ValueError: Received an invalid value for `units`, expected a positive integer.
```

Esses erros ocorrem porque o AutoKeras passou a ter problemas ao definir as camadas de saída quando usado com Keras 3+, tornando-o **não confiável para uso em ambiente de aprendizado**.

### O Optuna como alternativa superior

O **[Optuna](https://optuna.org/)** é um framework de otimização de hiperparâmetros moderno e amplamente adotado na indústria e academia. Ele oferece:

| Característica | AutoKeras | Optuna |
|---|---|---|
| Estabilidade com versões atuais | ❌ Problemas frequentes | ✅ Atualizado ativamente |
| Transparência no processo | ❌ Caixa-preta | ✅ Você define o espaço de busca |
| Flexibilidade de modelos | ❌ Só redes neurais | ✅ Qualquer modelo (sklearn, XGBoost, PyTorch...) |
| Algoritmos de busca inteligente | ✅ Sim | ✅ Sim (TPE, CMA-ES, etc.) |
| Visualizações e análises | ❌ Limitado | ✅ Dashboard interativo nativo |
| Curva de aprendizado | ❌ Mais difícil de customizar | ✅ Código explícito e didático |

> **Em resumo:** o Optuna nos dá o mesmo benefício do AutoML — busca automatizada pelos melhores hiperparâmetros — com muito mais controle, estabilidade e transparência. Você aprende *o que* está sendo otimizado, não apenas aceita um resultado de uma caixa-preta.

## **Instalação das bibliotecas**

In [ ]:
%%capture
!pip install optuna scikit-learn pandas

## **Import dos pacotes necessários**

In [ ]:
import os
import optuna
import pandas as pd
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score

# Silencia logs de progresso desnecessários do Optuna durante a busca
optuna.logging.set_verbosity(optuna.logging.WARNING)

## **Cópia do dataset**

In [ ]:
%%time
!rm -rf lectures-cdas-2023
!git clone https://github.com/renansantosmendes/lectures-cdas-2023.git

## **Leitura do dataset em um DataFrame do Pandas**

In [ ]:
data = pd.read_csv(os.path.join('lectures-cdas-2023', 'fetal_health.csv'))
print(f"Shape do dataset: {data.shape}")
data.head()

## **Processamento dos dados**

Nesta célula serão feitos os seguintes passos:
- Separação das variáveis de entrada (`X`) e saída (`y`)
- Padronização dos valores das variáveis de entrada com `StandardScaler`
- Separação em dados de treino e teste
- Ajuste dos labels: a coluna `fetal_health` contém valores `1`, `2` e `3`; convertemos para `0`, `1` e `2` para compatibilidade com scikit-learn

In [ ]:
X = data.drop(["fetal_health"], axis=1)
y = data["fetal_health"].astype(int) - 1  # Converte classes {1,2,3} para {0,1,2}

columns_names = list(X.columns)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_df = pd.DataFrame(X_scaled, columns=columns_names)

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.3, random_state=42
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Classes: {sorted(y.unique())} -> {dict(y.value_counts().sort_index())}")

## **Definindo a função objetivo para o Optuna**

Este é o coração do AutoML com Optuna. A ideia é simples:

1. Definimos uma **função objetivo** que recebe um objeto `trial`
2. Dentro dela, usamos `trial.suggest_*` para deixar o Optuna **escolher os hiperparâmetros automaticamente**
3. Treinamos o modelo com esses hiperparâmetros e retornamos uma métrica de avaliação
4. O Optuna repete esse processo várias vezes, **aprendendo quais combinações tendem a ser melhores** (algoritmo TPE — Tree-structured Parzen Estimator)

### Hiperparâmetros que serão otimizados:
- **n_layers**: número de camadas ocultas (1 a 3)
- **n_units_i**: número de neurônios em cada camada (32 a 256)
- **activation**: função de ativação (`relu`, `tanh`)
- **learning_rate_init**: taxa de aprendizado inicial
- **alpha**: coeficiente de regularização L2
- **batch_size**: tamanho do mini-batch

In [ ]:
def objective(trial):
    # --- Optuna sugere os hiperparâmetros ---

    # Número de camadas ocultas
    n_layers = trial.suggest_int("n_layers", 1, 3)

    # Número de neurônios por camada (definido individualmente para cada camada)
    hidden_layer_sizes = tuple(
        trial.suggest_int(f"n_units_l{i}", 32, 256, step=32)
        for i in range(n_layers)
    )

    # Função de ativação
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])

    # Taxa de aprendizado
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True)

    # Regularização L2
    alpha = trial.suggest_float("alpha", 1e-5, 1e-1, log=True)

    # Tamanho do mini-batch
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    # --- Criação e avaliação do modelo ---
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        learning_rate_init=learning_rate_init,
        alpha=alpha,
        batch_size=batch_size,
        max_iter=100,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10
    )

    # Usamos validação cruzada (3 folds) para uma avaliação mais robusta
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy", n_jobs=-1)

    return scores.mean()

## **Executando a busca com Optuna**

Agora criamos um **`study`** (estudo de otimização) e chamamos `.optimize()` para executar a busca.

- `n_trials=30`: o Optuna vai testar 30 combinações diferentes de hiperparâmetros
- `direction="maximize"`: queremos **maximizar** a acurácia
- O Optuna usa o algoritmo **TPE (Tree-structured Parzen Estimator)** por padrão, que é uma busca **bayesiana**: cada trial aprende com os anteriores para sugerir configurações mais promissoras

> 💡 **Dica:** Aumentar `n_trials` geralmente melhora o resultado, mas aumenta o tempo de execução.

In [ ]:
%%time

study = optuna.create_study(direction="maximize", study_name="fetal_health_mlp")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n" + "="*50)
print(f"Melhor acurácia (validação cruzada): {study.best_value:.4f}")
print("Melhores hiperparâmetros encontrados:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

## **Visualizando o histórico de otimização**

O Optuna oferece visualizações nativas para entender o comportamento da busca.

- **Optimization History**: mostra como a acurácia evoluiu ao longo dos trials
- **Param Importances**: mostra quais hiperparâmetros tiveram maior impacto no resultado

In [ ]:
# Histórico de otimização
fig1 = optuna.visualization.plot_optimization_history(study)
fig1.show()

In [ ]:
# Importância dos hiperparâmetros
fig2 = optuna.visualization.plot_param_importances(study)
fig2.show()

## **Treinando o modelo final com os melhores hiperparâmetros**

Com a busca concluída, usamos os melhores hiperparâmetros encontrados para treinar o **modelo final** em todos os dados de treino.

In [ ]:
best_params = study.best_params

# Reconstrói hidden_layer_sizes a partir dos parâmetros salvos
n_layers = best_params["n_layers"]
hidden_layer_sizes = tuple(
    best_params[f"n_units_l{i}"] for i in range(n_layers)
)

best_model = MLPClassifier(
    hidden_layer_sizes=hidden_layer_sizes,
    activation=best_params["activation"],
    learning_rate_init=best_params["learning_rate_init"],
    alpha=best_params["alpha"],
    batch_size=best_params["batch_size"],
    max_iter=200,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=10
)

best_model.fit(X_train, y_train)

print("Arquitetura do melhor modelo:")
print(f"  Camadas ocultas: {hidden_layer_sizes}")
print(f"  Ativação: {best_params['activation']}")
print(f"  Taxa de aprendizado: {best_params['learning_rate_init']:.6f}")
print(f"  Alpha (L2): {best_params['alpha']:.6f}")
print(f"  Batch size: {best_params['batch_size']}")

## **Avaliando o modelo final com os dados de teste**

Agora avaliamos o modelo com os dados de teste — que **não foram usados em nenhum momento** durante a busca de hiperparâmetros, garantindo uma avaliação justa.

In [ ]:
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Acurácia no conjunto de teste: {accuracy:.4f}")
print()

# Relatório completo por classe
# As classes originais eram: 1=Normal, 2=Suspeito, 3=Patológico
target_names = ["Normal (0)", "Suspeito (1)", "Patologico (2)"]
print("Relatório de classificação:")
print(classification_report(y_test, y_pred, target_names=target_names))

## **Resumo do que aprendemos**

Neste notebook, aplicamos **AutoML com Optuna** para otimizar automaticamente uma rede neural MLP no problema de classificação da saúde fetal.

### Fluxo do processo:

```
1. Definir o espaço de busca (quais hiperparâmetros e seus intervalos)
       ↓
2. Criar a função objetivo (treinar e avaliar com os parâmetros sugeridos)
       ↓
3. Executar o study (Optuna testa N combinações de forma inteligente)
       ↓
4. Recuperar os melhores parâmetros e treinar o modelo final
       ↓
5. Avaliar no conjunto de teste
```

### Por que isso é AutoML?

Porque **você não precisou testar manualmente** cada combinação de hiperparâmetros. O Optuna fez isso de forma automatizada e inteligente, usando o histórico de trials anteriores para guiar a busca — exatamente o princípio do AutoML.

> **Próximos passos sugeridos:**
> - Aumentar `n_trials` para uma busca mais ampla
> - Incluir outros tipos de modelo na busca (Random Forest, XGBoost, SVM)
> - Explorar o `optuna.visualization.plot_contour` para entender interações entre parâmetros